# 00 - Environment and Reproducibility

Establish the execution contract shared by all MovieLens-1M cold-start
experiments. This notebook inspects the runtime, audits dependencies, validates
accelerator support, configures deterministic execution, and records
provenance. It does not download data, create splits, or train models.

## Scope

- Dataset: MovieLens-1M only.
- Baselines: LightGCN and EmerG only.
- Proposed model: Differentiable Graph Diffusion (DGD).
- Execution targets: local Jupyter, Google Colab, and private Kaggle T4 kernels.
- Network policy: offline by default.

A successful run reports environment readiness without changing project data
or creating visible repository artifacts. Runtime caches stay in the notebook
workspace.

## Runtime Context

Use environment variables only as explicit overrides. Otherwise, discover the
execution platform and project root from the current working directory.

In [2]:
import importlib.util
import subprocess
import sys

def ensure_package(module_name: str, package_name: str) -> None:
    if importlib.util.find_spec(module_name) is None:
        subprocess.check_call([
            sys.executable,
            "-m",
            "pip",
            "install",
            "-q",
            package_name,
        ])

ensure_package("torch_geometric", "torch-geometric")
ensure_package("nni", "nni")

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 64.4/64.4 kB 2.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 23.3 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.4/48.4 kB 1.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.4/61.4 MB 30.7 MB/s eta 0:00:00


ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
pytensor 2.38.2 requires filelock>=3.15, but you have filelock 3.11.0 which is incompatible.


In [2]:
from __future__ import annotations

import hashlib
import importlib
import json
import os
import platform
import random
import shutil
import subprocess
import sys
from dataclasses import asdict, dataclass
from datetime import datetime, timezone
from importlib import metadata, util
from pathlib import Path
from typing import Any, Iterable

from IPython.display import Markdown, display

# This must be set before Torch initializes a CUDA context.
VALID_CUBLAS_CONFIGURATIONS = {":16:8", ":4096:8"}
CUBLAS_CONFIG_AT_ENTRY = os.environ.get("CUBLAS_WORKSPACE_CONFIG")
if CUBLAS_CONFIG_AT_ENTRY not in VALID_CUBLAS_CONFIGURATIONS:
    os.environ["CUBLAS_WORKSPACE_CONFIG"] = ":4096:8"


def show_records(
    records: Iterable[dict[str, Any]], columns: list[str] | None = None
) -> None:
    """Display records as a DataFrame when pandas is available."""
    rows = list(records)
    if not rows:
        display([])
        return

    if util.find_spec("pandas") is None:
        display(rows)
        return

    try:
        pandas = importlib.import_module("pandas")
    except Exception:
        display(rows)
        return
    table = pandas.DataFrame(rows)
    display(table if columns is None else table.reindex(columns=columns))

In [3]:
def detect_execution_context() -> str:
    if os.environ.get("KAGGLE_KERNEL_RUN_TYPE") or Path("/kaggle/input").exists():
        return "kaggle"
    if os.environ.get("COLAB_RELEASE_TAG") or "google.colab" in sys.modules:
        return "colab"
    return "local"


def discover_project_root(start: Path) -> tuple[Path, str]:
    override = os.environ.get("COLDSTART_PROJECT_ROOT")
    if override:
        return Path(override).expanduser().resolve(), "COLDSTART_PROJECT_ROOT"

    for candidate in (start, *start.parents):
        if (candidate / ".git").exists():
            return candidate.resolve(), ".git marker"
        if (candidate / "notebooks").is_dir() and (candidate / "README.md").exists():
            return candidate.resolve(), "repository markers"

    return start.resolve(), "working-directory fallback"


def configured_path(variable: str, default: Path, base: Path) -> Path:
    value = Path(os.environ.get(variable, str(default))).expanduser()
    if not value.is_absolute():
        value = base / value
    return value.resolve()

In [4]:
EXECUTION_CONTEXT = detect_execution_context()
CURRENT_WORKING_DIRECTORY = Path.cwd().resolve()
PROJECT_ROOT, PROJECT_ROOT_SOURCE = discover_project_root(CURRENT_WORKING_DIRECTORY)

if EXECUTION_CONTEXT == "kaggle":
    default_workspace = Path("/kaggle/working")
    default_input = Path("/kaggle/input")
elif EXECUTION_CONTEXT == "colab":
    default_workspace = Path("/content")
    default_input = Path("/content/data")
else:
    default_workspace = PROJECT_ROOT / ".notebook"
    default_input = PROJECT_ROOT / "data"

WORKSPACE_ROOT = configured_path("COLDSTART_WORKSPACE_ROOT", default_workspace, PROJECT_ROOT)
INPUT_ROOT = configured_path("COLDSTART_INPUT_ROOT", default_input, PROJECT_ROOT)
ARTIFACT_ROOT = configured_path(
    "COLDSTART_ARTIFACT_ROOT", WORKSPACE_ROOT / "artifacts", PROJECT_ROOT
)
MATPLOTLIB_CACHE_ROOT = WORKSPACE_ROOT / "cache" / "matplotlib"
os.environ["MPLCONFIGDIR"] = str(MATPLOTLIB_CACHE_ROOT)

RUNTIME_CONTEXT = {
    "execution_context": EXECUTION_CONTEXT,
    "python_version": platform.python_version(),
    "python_executable": sys.executable,
    "platform": platform.platform(),
    "machine": platform.machine(),
    "working_directory": str(CURRENT_WORKING_DIRECTORY),
    "project_root": str(PROJECT_ROOT),
    "project_root_source": PROJECT_ROOT_SOURCE,
    "input_root": str(INPUT_ROOT),
    "workspace_root": str(WORKSPACE_ROOT),
    "artifact_root": str(ARTIFACT_ROOT),
}

show_records([RUNTIME_CONTEXT])

,execution_context,python_version,python_executable,platform,machine,working_directory,project_root,project_root_source,input_root,workspace_root,artifact_root
0,local,3.12.3,/workspace/.venv/bin/python,Linux-6.8.0-52-generic-x86_64-with-glibc2.39,x86_64,/workspace/HungPH/coldstart-recsys/notebooks,/workspace/HungPH/coldstart-recsys,.git marker,/workspace/HungPH/coldstart-recsys/data,/workspace/HungPH/coldstart-recsys/.notebook,/workspace/HungPH/coldstart-recsys/.notebook/a...


## Dependency Contract

Required packages support the proposed LightGCN, EmerG, DGD, analysis, and PyG
workflows. Optional packages are retained only for legacy code or notebook
ergonomics. This notebook audits the environment but deliberately performs no
package installation.

In [5]:
def dependency(
    group: str,
    module: str,
    distribution: str,
    required: bool,
    purpose: str,
) -> dict[str, Any]:
    return {
        "group": group,
        "module": module,
        "distribution": distribution,
        "required": required,
        "purpose": purpose,
    }


DEPENDENCIES = (
    dependency("core", "numpy", "numpy", True, "numerical operations and sampling"),
    dependency("core", "pandas", "pandas", True, "tabular data and run records"),
    dependency("core", "scipy", "scipy", True, "sparse graph construction"),
    dependency(
        "core", "sklearn", "scikit-learn", True, "metrics and preprocessing"
    ),
    dependency("graph-ml", "torch", "torch", True, "modeling and sparse tensors"),
    dependency(
        "graph-ml", "torch_geometric", "torch-geometric", True, "PyG message passing"
    ),
    dependency("reporting", "matplotlib", "matplotlib", True, "figures"),
    dependency("tooling", "tqdm", "tqdm", True, "progress reporting"),
    dependency("optional", "ipywidgets", "ipywidgets", False, "notebook widgets"),
    dependency("legacy", "networkx", "networkx", True, "current EmerG imports"),
    dependency("legacy", "nltk", "nltk", False, "title preprocessing"),
    dependency("legacy", "nni", "nni", True, "current EmerG entry point"),
)

In [6]:
def audit_dependency(specification: dict[str, Any]) -> dict[str, Any]:
    discoverable = util.find_spec(specification["module"]) is not None
    import_error = None
    importable = discoverable
    if discoverable and specification["required"]:
        try:
            importlib.import_module(specification["module"])
        except Exception as error:
            importable = False
            import_error = repr(error)

    try:
        version = metadata.version(specification["distribution"])
    except metadata.PackageNotFoundError:
        version = None

    return {
        **specification,
        "discoverable": discoverable,
        "importable": importable,
        "import_error": import_error,
        "version": version,
        "status": "PASS" if importable else ("FAIL" if specification["required"] else "WARN"),
    }


DEPENDENCY_AUDIT = [audit_dependency(specification) for specification in DEPENDENCIES]
MISSING_REQUIRED = [
    record["distribution"]
    for record in DEPENDENCY_AUDIT
    if record["required"] and not record["importable"]
]

show_records(
    DEPENDENCY_AUDIT,
    [
        "group",
        "distribution",
        "required",
        "discoverable",
        "importable",
        "version",
        "status",
        "purpose",
        "import_error",
    ],
)

,group,distribution,required,discoverable,importable,version,status,purpose,import_error
0,core,numpy,True,True,True,2.4.4,PASS,numerical operations and sampling,None
1,core,pandas,True,True,True,3.0.3,PASS,tabular data and run records,None
2,core,scipy,True,False,False,NaN,FAIL,sparse graph construction,None
3,core,scikit-learn,True,False,False,NaN,FAIL,metrics and preprocessing,None
4,graph-ml,torch,True,True,True,2.11.0+cu128,PASS,modeling and sparse tensors,None
5,graph-ml,torch-geometric,True,False,False,NaN,FAIL,PyG message passing,None
6,reporting,matplotlib,True,False,False,NaN,FAIL,figures,None
7,tooling,tqdm,True,False,False,NaN,FAIL,progress reporting,None
8,optional,ipywidgets,False,False,False,NaN,WARN,notebook widgets,None
9,legacy,networkx,True,True,True,3.6.1,PASS,current EmerG imports,None


In [7]:
ROOT_MANIFEST_NAMES = (
    "pyproject.toml",
    "requirements.txt",
    "environment.yml",
    "environment.yaml",
    "uv.lock",
    "poetry.lock",
)
ROOT_DEPENDENCY_MANIFESTS = [
    PROJECT_ROOT / name for name in ROOT_MANIFEST_NAMES if (PROJECT_ROOT / name).exists()
]

def optional_import(module: str) -> Any | None:
    try:
        return importlib.import_module(module)
    except Exception:
        return None


numpy = optional_import("numpy")
torch = optional_import("torch")

print("Missing required packages:", MISSING_REQUIRED or "none")
print(
    "Root dependency manifests:",
    [path.name for path in ROOT_DEPENDENCY_MANIFESTS] or "none (version locking is pending)",
)

Missing required packages: ['scipy', 'scikit-learn', 'torch-geometric', 'matplotlib', 'tqdm', 'nni']
Root dependency manifests: none (version locking is pending)


## Accelerator Contract

`COLDSTART_DEVICE` can explicitly request a device. Without an override, the
order is CUDA, Apple MPS, then CPU. CUDA device indices are validated before
use, which avoids the legacy EmerG assumption that `cuda:1` exists.

In [8]:
def select_device(torch_module: Any) -> tuple[Any | None, str, str | None]:
    if torch_module is None:
        return None, "unavailable", "PyTorch is not installed"

    override = os.environ.get("COLDSTART_DEVICE")
    if override:
        try:
            device = torch_module.device(override)
            if device.type == "cuda":
                if not torch_module.cuda.is_available():
                    raise ValueError("CUDA is not available")
                index = 0 if device.index is None else device.index
                if index >= torch_module.cuda.device_count():
                    raise ValueError(f"CUDA device index {index} does not exist")
            if device.type == "mps" and not (
                hasattr(torch_module.backends, "mps")
                and torch_module.backends.mps.is_available()
            ):
                raise ValueError("MPS is not available")
            return device, "COLDSTART_DEVICE override", None
        except (RuntimeError, ValueError) as error:
            return None, "invalid COLDSTART_DEVICE override", str(error)

    if torch_module.cuda.is_available():
        return torch_module.device("cuda:0"), "first available CUDA device", None

    if (
        hasattr(torch_module.backends, "mps")
        and torch_module.backends.mps.is_available()
    ):
        return torch_module.device("mps"), "available Apple MPS device", None

    return torch_module.device("cpu"), "CPU fallback", None


DEVICE, DEVICE_SELECTION_REASON, DEVICE_SELECTION_ERROR = select_device(torch)

DEVICE_REPORT = {
    "selected_device": str(DEVICE) if DEVICE is not None else None,
    "selection_reason": DEVICE_SELECTION_REASON,
    "selection_error": DEVICE_SELECTION_ERROR,
    "torch_version": getattr(torch, "__version__", None),
    "torch_cuda_build": getattr(getattr(torch, "version", None), "cuda", None),
    "cuda_available": bool(torch and torch.cuda.is_available()),
    "cuda_device_count": torch.cuda.device_count() if torch and torch.cuda.is_available() else 0,
    "cudnn_version": (
        torch.backends.cudnn.version()
        if torch and torch.cuda.is_available() and torch.backends.cudnn.is_available()
        else None
    ),
    "mps_available": bool(
        torch
        and hasattr(torch.backends, "mps")
        and torch.backends.mps.is_available()
    ),
}

if torch and torch.cuda.is_available():
    selected_index = (
        DEVICE.index
        if DEVICE is not None and DEVICE.type == "cuda" and DEVICE.index is not None
        else 0
    )
    properties = torch.cuda.get_device_properties(selected_index)
    DEVICE_REPORT.update(
        {
            "gpu_name": properties.name,
            "gpu_capability": ".".join(map(str, torch.cuda.get_device_capability(selected_index))),
            "gpu_memory_gib": round(properties.total_memory / 1024**3, 2),
        }
    )

if EXECUTION_CONTEXT != "kaggle":
    KAGGLE_T4_STATUS = "WARN"
    KAGGLE_T4_DETAIL = "verified only when running on Kaggle"
elif DEVICE is None or DEVICE.type != "cuda":
    KAGGLE_T4_STATUS = "FAIL"
    KAGGLE_T4_DETAIL = "Kaggle execution requires an available CUDA device"
elif "T4" not in DEVICE_REPORT.get("gpu_name", ""):
    KAGGLE_T4_STATUS = "FAIL"
    KAGGLE_T4_DETAIL = f"expected T4, found {DEVICE_REPORT.get('gpu_name', 'unknown GPU')}"
else:
    KAGGLE_T4_STATUS = "PASS"
    KAGGLE_T4_DETAIL = DEVICE_REPORT["gpu_name"]

show_records([DEVICE_REPORT])

,selected_device,selection_reason,selection_error,torch_version,torch_cuda_build,cuda_available,cuda_device_count,cudnn_version,mps_available,gpu_name,gpu_capability,gpu_memory_gib
0,cuda:0,first available CUDA device,None,2.11.0+cu128,12.8,True,1,91900,False,NVIDIA GeForce RTX 5080,12.0,15.48


In [9]:
def run_device_smoke_checks(torch_module: Any, device: Any) -> list[dict[str, str]]:
    results: list[dict[str, str]] = []

    if torch_module is None or device is None:
        reason = DEVICE_SELECTION_ERROR or "PyTorch or a valid device is unavailable"
        return [
            {"check": name, "status": "FAIL", "detail": reason}
            for name in ("torch_dense", "torch_sparse", "pyg_message_passing")
        ]

    try:
        torch_module.manual_seed(0)
        features = torch_module.arange(6, dtype=torch_module.float32, device=device).reshape(3, 2)
        dense_result = features @ features.T
        passed = dense_result.shape == (3, 3) and bool(torch_module.isfinite(dense_result).all())
        results.append(
            {
                "check": "torch_dense",
                "status": "PASS" if passed else "FAIL",
                "detail": f"shape={tuple(dense_result.shape)}, device={dense_result.device}",
            }
        )
    except Exception as error:  # Runtime capability errors vary by backend.
        results.append({"check": "torch_dense", "status": "FAIL", "detail": repr(error)})

    try:
        indices = torch_module.tensor(
            [[0, 1, 1], [1, 0, 1]], dtype=torch_module.long, device=device
        )
        values = torch_module.tensor([1.0, 1.0, 1.0], device=device)
        adjacency = torch_module.sparse_coo_tensor(indices, values, size=(2, 2)).coalesce()
        sparse_result = torch_module.sparse.mm(
            adjacency, torch_module.ones((2, 2), device=device)
        )
        passed = sparse_result.shape == (2, 2) and bool(torch_module.isfinite(sparse_result).all())
        results.append(
            {
                "check": "torch_sparse",
                "status": "PASS" if passed else "FAIL",
                "detail": f"sparse matrix multiplication on {sparse_result.device}",
            }
        )
    except Exception as error:
        results.append({"check": "torch_sparse", "status": "FAIL", "detail": repr(error)})

    try:
        geometric_nn = importlib.import_module("torch_geometric.nn")
        edge_index = torch_module.tensor(
            [[0, 1, 1, 2], [1, 0, 2, 1]], dtype=torch_module.long, device=device
        )
        node_features = torch_module.ones((3, 2), dtype=torch_module.float32, device=device)
        layer = geometric_nn.GCNConv(2, 2).to(device).eval()
        with torch_module.no_grad():
            pyg_result = layer(node_features, edge_index)
        passed = pyg_result.shape == (3, 2) and bool(torch_module.isfinite(pyg_result).all())
        results.append(
            {
                "check": "pyg_message_passing",
                "status": "PASS" if passed else "FAIL",
                "detail": f"GCNConv shape={tuple(pyg_result.shape)}, device={pyg_result.device}",
            }
        )
    except Exception as error:
        results.append(
            {"check": "pyg_message_passing", "status": "FAIL", "detail": repr(error)}
        )

    return results


SMOKE_CHECKS = run_device_smoke_checks(torch, DEVICE)
show_records(SMOKE_CHECKS, ["check", "status", "detail"])

/tmp/ipykernel_13057/626686349.py:31: UserWarning: Sparse invariant checks are implicitly disabled. Memory errors (e.g. SEGFAULT) will occur when operating on a sparse tensor which violates the invariants, but checks incur performance overhead. To silence this warning, explicitly opt in or out. See `torch.sparse.check_sparse_tensor_invariants.__doc__` for guidance.  (Triggered internally at /pytorch/aten/src/ATen/Context.cpp:760.)
  adjacency = torch_module.sparse_coo_tensor(indices, values, size=(2, 2)).coalesce()


,check,status,detail
0,torch_dense,PASS,"shape=(3, 3), device=cuda:0"
1,torch_sparse,PASS,sparse matrix multiplication on cuda:0
2,pyg_message_passing,PASS,"GCNConv shape=(3, 2), device=cuda:0"


## Shared Configuration

Keep scope, path, and reproducibility settings explicit and immutable. Dataset
schema, split, model, training, and evaluation settings belong to later
notebooks and the future shared package.

In [10]:
@dataclass(frozen=True)
class ScopeConfig:
    dataset: str = "MovieLens-1M"
    baselines: tuple[str, ...] = ("LightGCN", "EmerG")
    proposed_model: str = "DGD"
    internet_required: bool = False


@dataclass(frozen=True)
class PathConfig:
    project_root: Path
    input_root: Path
    workspace_root: Path
    artifact_root: Path


@dataclass(frozen=True)
class ReproducibilityConfig:
    seed: int
    deterministic_algorithms: bool
    device: str


DEFAULT_SEED_INPUT = os.environ.get("COLDSTART_SEED", "2025")
SEED_CONFIGURATION_ERROR = None
try:
    parsed_seed = int(DEFAULT_SEED_INPUT)
    if not 0 <= parsed_seed < 2**32:
        raise ValueError("seed must satisfy 0 <= seed < 2**32")
except ValueError as error:
    DEFAULT_SEED = 2025
    SEED_CONFIGURATION_ERROR = str(error)
else:
    DEFAULT_SEED = parsed_seed

SCOPE_CONFIG = ScopeConfig()
PATH_CONFIG = PathConfig(PROJECT_ROOT, INPUT_ROOT, WORKSPACE_ROOT, ARTIFACT_ROOT)
REPRODUCIBILITY_CONFIG = ReproducibilityConfig(
    seed=DEFAULT_SEED,
    deterministic_algorithms=True,
    device=str(DEVICE) if DEVICE is not None else "unavailable",
)

show_records(
    [
        {"contract": "scope", **asdict(SCOPE_CONFIG)},
        {"contract": "paths", **{key: str(value) for key, value in asdict(PATH_CONFIG).items()}},
        {"contract": "reproducibility", **asdict(REPRODUCIBILITY_CONFIG)},
    ]
)

,contract,dataset,baselines,proposed_model,internet_required,project_root,input_root,workspace_root,artifact_root,seed,deterministic_algorithms,device
0,scope,MovieLens-1M,"(LightGCN, EmerG)",DGD,False,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,paths,NaN,NaN,NaN,NaN,/workspace/HungPH/coldstart-recsys,/workspace/HungPH/coldstart-recsys/data,/workspace/HungPH/coldstart-recsys/.notebook,/workspace/HungPH/coldstart-recsys/.notebook/a...,NaN,NaN,NaN
2,reproducibility,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2025.0,True,cuda:0


## Reproducibility

Seed every available random-number generator and configure deterministic Torch
behavior. `PYTHONHASHSEED` affects child processes after it is set; if it was
absent at kernel startup, restart the kernel with the reported value for full
interpreter-level hash reproducibility.

In [11]:
def seed_everything(seed: int, deterministic: bool = True) -> dict[str, Any]:
    hash_seed_at_entry = os.environ.get("PYTHONHASHSEED")
    os.environ["PYTHONHASHSEED"] = str(seed)

    random.seed(seed)
    if numpy is not None:
        numpy.random.seed(seed)

    settings: dict[str, Any] = {
        "seed": seed,
        "python_hash_seed": os.environ["PYTHONHASHSEED"],
        "python_hash_seed_at_entry": hash_seed_at_entry,
        "python_hash_seed_matched_at_entry": hash_seed_at_entry == str(seed),
        "cublas_workspace_config": os.environ["CUBLAS_WORKSPACE_CONFIG"],
        "deterministic_algorithms": deterministic,
    }

    if torch is None:
        settings["torch_seeded"] = False
        return settings

    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

    torch.use_deterministic_algorithms(deterministic)

    if hasattr(torch.backends, "cudnn"):
        torch.backends.cudnn.deterministic = deterministic
        torch.backends.cudnn.benchmark = False
        if hasattr(torch.backends.cudnn, "allow_tf32"):
            torch.backends.cudnn.allow_tf32 = False
    if hasattr(torch.backends, "cuda") and hasattr(torch.backends.cuda, "matmul"):
        torch.backends.cuda.matmul.allow_tf32 = False

    settings.update(
        {
            "torch_seeded": True,
            "torch_deterministic_algorithms": torch.are_deterministic_algorithms_enabled(),
            "cudnn_deterministic": getattr(torch.backends.cudnn, "deterministic", None),
            "cudnn_benchmark": getattr(torch.backends.cudnn, "benchmark", None),
        }
    )
    return settings


def make_dataloader_generator(seed: int) -> Any | None:
    if torch is None:
        return None
    generator = torch.Generator()
    generator.manual_seed(seed)
    return generator


def seed_dataloader_worker(worker_id: int) -> None:
    if torch is None or numpy is None:
        return
    worker_seed = torch.initial_seed() % 2**32
    numpy.random.seed(worker_seed)
    random.seed(worker_seed)


SEED_SETTINGS = seed_everything(
    REPRODUCIBILITY_CONFIG.seed,
    REPRODUCIBILITY_CONFIG.deterministic_algorithms,
)
DATALOADER_GENERATOR = make_dataloader_generator(REPRODUCIBILITY_CONFIG.seed)
show_records([SEED_SETTINGS])

,seed,python_hash_seed,python_hash_seed_at_entry,python_hash_seed_matched_at_entry,cublas_workspace_config,deterministic_algorithms,torch_seeded,torch_deterministic_algorithms,cudnn_deterministic,cudnn_benchmark
0,2025,2025,None,False,:4096:8,True,True,True,True,False


In [12]:
def random_snapshot() -> dict[str, list[float]]:
    snapshot = {"python": [random.random() for _ in range(3)]}
    if numpy is not None:
        snapshot["numpy"] = numpy.random.random(3).tolist()
    if torch is not None and DEVICE is not None:
        features = torch.rand((3, 2), device=DEVICE)
        snapshot["torch_random"] = features.flatten().cpu().tolist()
        snapshot["torch_dense"] = (features @ features.T).flatten().cpu().tolist()

        geometric_nn = optional_import("torch_geometric.nn")
        if geometric_nn is not None:
            edge_index = torch.tensor(
                [[0, 1, 1, 2], [1, 0, 2, 1]], dtype=torch.long, device=DEVICE
            )
            layer = geometric_nn.GCNConv(2, 2).to(DEVICE).eval()
            with torch.no_grad():
                pyg_result = layer(features, edge_index)
            snapshot["pyg_message_passing"] = pyg_result.flatten().cpu().tolist()
    return snapshot


REPEATABILITY_ERROR = None
try:
    seed_everything(DEFAULT_SEED, deterministic=True)
    FIRST_SNAPSHOT = random_snapshot()
    seed_everything(DEFAULT_SEED, deterministic=True)
    SECOND_SNAPSHOT = random_snapshot()
    REPEATABILITY_CONFIRMED = FIRST_SNAPSHOT == SECOND_SNAPSHOT
except Exception as error:
    FIRST_SNAPSHOT = {}
    SECOND_SNAPSHOT = {}
    REPEATABILITY_CONFIRMED = False
    REPEATABILITY_ERROR = repr(error)
finally:
    # Leave the notebook in the documented initial RNG state.
    seed_everything(DEFAULT_SEED, deterministic=True)

show_records(
    [
        {
            "repeatability_confirmed": REPEATABILITY_CONFIRMED,
            "error": REPEATABILITY_ERROR,
            "first_snapshot": FIRST_SNAPSHOT,
            "second_snapshot": SECOND_SNAPSHOT,
        }
    ]
)

,repeatability_confirmed,error,first_snapshot,second_snapshot
0,True,None,"{'python': [0.5577521702848538, 0.645844912808...","{'python': [0.5577521702848538, 0.645844912808..."


## Provenance Record

Capture only non-secret execution metadata. The record remains in memory in
this notebook; the future experiment runner will persist one record per run.

In [13]:
def git_command(*arguments: str) -> str | None:
    if shutil.which("git") is None or not (PROJECT_ROOT / ".git").exists():
        return None
    try:
        result = subprocess.run(
            ["git", *arguments],
            cwd=PROJECT_ROOT,
            check=False,
            capture_output=True,
            text=True,
            timeout=5,
        )
    except (OSError, subprocess.TimeoutExpired):
        return None
    return result.stdout.strip() if result.returncode == 0 else None


def json_ready(value: Any) -> Any:
    if isinstance(value, Path):
        return str(value)
    if isinstance(value, dict):
        return {str(key): json_ready(item) for key, item in value.items()}
    if isinstance(value, (list, tuple)):
        return [json_ready(item) for item in value]
    return value


def sha256_file(path: Path) -> str:
    digest = hashlib.sha256()
    with path.open("rb") as stream:
        for chunk in iter(lambda: stream.read(1024 * 1024), b""):
            digest.update(chunk)
    return digest.hexdigest()


NOTEBOOK_NAME = "00_environment_and_reproducibility.ipynb"
NOTEBOOK_CANDIDATES = (
    PROJECT_ROOT / "notebooks" / NOTEBOOK_NAME,
    CURRENT_WORKING_DIRECTORY / NOTEBOOK_NAME,
)
NOTEBOOK_SOURCE_PATH = next((path for path in NOTEBOOK_CANDIDATES if path.is_file()), None)
NOTEBOOK_SOURCE_HASH = sha256_file(NOTEBOOK_SOURCE_PATH) if NOTEBOOK_SOURCE_PATH else None

CONFIGURATION_PAYLOAD = json_ready(
    {
        "scope": asdict(SCOPE_CONFIG),
        "paths": asdict(PATH_CONFIG),
        "reproducibility": asdict(REPRODUCIBILITY_CONFIG),
    }
)
CONFIGURATION_JSON = json.dumps(CONFIGURATION_PAYLOAD, sort_keys=True, separators=(",", ":"))
CONFIGURATION_HASH = hashlib.sha256(CONFIGURATION_JSON.encode("utf-8")).hexdigest()

GIT_STATUS = git_command("status", "--porcelain")
PROVENANCE = {
    "captured_at_utc": datetime.now(timezone.utc).isoformat(),
    "notebook": NOTEBOOK_NAME,
    "notebook_source_sha256": NOTEBOOK_SOURCE_HASH,
    "runtime": RUNTIME_CONTEXT,
    "device": DEVICE_REPORT,
    "dependency_versions": {
        record["distribution"]: record["version"] for record in DEPENDENCY_AUDIT
    },
    "git_commit": git_command("rev-parse", "HEAD"),
    "git_dirty": None if GIT_STATUS is None else bool(GIT_STATUS),
    "safe_environment": {
        name: os.environ.get(name)
        for name in (
            "PYTHONHASHSEED",
            "CUBLAS_WORKSPACE_CONFIG",
            "MPLCONFIGDIR",
            "COLDSTART_DEVICE",
            "KAGGLE_KERNEL_RUN_TYPE",
            "COLAB_RELEASE_TAG",
        )
    },
    "configuration_hash_sha256": CONFIGURATION_HASH,
}

display(PROVENANCE)

{'captured_at_utc': '2026-07-16T18:31:23.659032+00:00',
 'notebook': '00_environment_and_reproducibility.ipynb',
 'notebook_source_sha256': '3631cc2ddd8dd6d3ec05f551821c83eb16f1fbd0111fbaf47f1f1ff964e5c482',
 'runtime': {'execution_context': 'local',
  'python_version': '3.12.3',
  'python_executable': '/workspace/.venv/bin/python',
  'platform': 'Linux-6.8.0-52-generic-x86_64-with-glibc2.39',
  'machine': 'x86_64',
  'working_directory': '/workspace/HungPH/coldstart-recsys/notebooks',
  'project_root': '/workspace/HungPH/coldstart-recsys',
  'project_root_source': '.git marker',
  'input_root': '/workspace/HungPH/coldstart-recsys/data',
  'workspace_root': '/workspace/HungPH/coldstart-recsys/.notebook',
  'artifact_root': '/workspace/HungPH/coldstart-recsys/.notebook/artifacts'},
 'device': {'selected_device': 'cuda:0',
  'selection_reason': 'first available CUDA device',
  'selection_error': None,
  'torch_version': '2.11.0+cu128',
  'torch_cuda_build': '12.8',
  'cuda_available': Tr

## Planned Shared Code

Reusable implementation should move outside notebooks before model work begins:

- `src/config.py`: typed experiment configuration and path contracts.
- `src/data/ml1m.py`: MovieLens-1M acquisition, parsing, and validation.
- `src/data/cold_start.py`: split construction and warm-up phases.
- `src/models/lightgcn.py`: LightGCN baseline.
- `src/models/emerg.py`: EmerG baseline.
- `src/models/dgd.py`: proposed DGD components.
- `src/training.py`: shared training, checkpointing, and tuning loops.
- `src/evaluation.py`: metrics, threshold selection, and multi-seed summaries.
- `src/reporting.py`: result tables and publication-ready figures.
- `src/utils.py`: reproducibility and runtime helpers.
- `tests/`: data-contract, leakage, model-shape, gradient, and metric tests.

The helper functions in this notebook are the executable contract for those
future modules, not a replacement for them.

## Artifact Contract

Raw inputs are read-only. Every generated artifact belongs under the writable
artifact root, which is `.notebook/artifacts` locally and `/kaggle/working/artifacts`
on Kaggle unless explicitly overridden. This cell defines paths but creates
nothing.

In [14]:
ARTIFACT_PATHS = {
    "raw_input": INPUT_ROOT,
    "processed_data": ARTIFACT_ROOT / "processed",
    "split_manifests": ARTIFACT_ROOT / "splits",
    "checkpoints": ARTIFACT_ROOT / "checkpoints",
    "run_records": ARTIFACT_ROOT / "runs",
    "metrics": ARTIFACT_ROOT / "metrics",
    "tables": ARTIFACT_ROOT / "tables",
    "figures": ARTIFACT_ROOT / "figures",
}


def is_within(path: Path, parent: Path) -> bool:
    try:
        path.resolve().relative_to(parent.resolve())
        return True
    except ValueError:
        return False


def nearest_existing_parent(path: Path) -> Path:
    candidate = path.resolve()
    while not candidate.exists() and candidate != candidate.parent:
        candidate = candidate.parent
    return candidate


ARTIFACT_EXISTING_PARENT = nearest_existing_parent(ARTIFACT_ROOT)

PATH_CHECKS = [
    {
        "check": "project_root_exists",
        "status": "PASS" if PROJECT_ROOT.is_dir() else "FAIL",
        "detail": str(PROJECT_ROOT),
    },
    {
        "check": "workspace_root_exists",
        "status": "PASS" if WORKSPACE_ROOT.is_dir() else "FAIL",
        "detail": str(WORKSPACE_ROOT),
    },
    {
        "check": "artifact_root_writable",
        "status": (
            "PASS"
            if ARTIFACT_EXISTING_PARENT.is_dir()
            and os.access(ARTIFACT_EXISTING_PARENT, os.W_OK | os.X_OK)
            and (not ARTIFACT_ROOT.exists() or ARTIFACT_ROOT.is_dir())
            else "FAIL"
        ),
        "detail": f"nearest existing parent: {ARTIFACT_EXISTING_PARENT}",
    },
    {
        "check": "input_root_available",
        "status": "PASS" if INPUT_ROOT.is_dir() else "WARN",
        "detail": str(INPUT_ROOT),
    },
    {
        "check": "artifacts_outside_read_only_input",
        "status": "FAIL" if is_within(ARTIFACT_ROOT, INPUT_ROOT) else "PASS",
        "detail": str(ARTIFACT_ROOT),
    },
]

show_records(
    [
        {"artifact": name, "path": str(path), "created": path.exists()}
        for name, path in ARTIFACT_PATHS.items()
    ]
)
show_records(PATH_CHECKS, ["check", "status", "detail"])

,artifact,path,created
0,raw_input,/workspace/HungPH/coldstart-recsys/data,True
1,processed_data,/workspace/HungPH/coldstart-recsys/.notebook/a...,True
2,split_manifests,/workspace/HungPH/coldstart-recsys/.notebook/a...,False
3,checkpoints,/workspace/HungPH/coldstart-recsys/.notebook/a...,False
4,run_records,/workspace/HungPH/coldstart-recsys/.notebook/a...,False
5,metrics,/workspace/HungPH/coldstart-recsys/.notebook/a...,False
6,tables,/workspace/HungPH/coldstart-recsys/.notebook/a...,False
7,figures,/workspace/HungPH/coldstart-recsys/.notebook/a...,False


,check,status,detail
0,project_root_exists,PASS,/workspace/HungPH/coldstart-recsys
1,workspace_root_exists,PASS,/workspace/HungPH/coldstart-recsys/.notebook
2,artifact_root_writable,PASS,nearest existing parent: /workspace/HungPH/col...
3,input_root_available,PASS,/workspace/HungPH/coldstart-recsys/data
4,artifacts_outside_read_only_input,PASS,/workspace/HungPH/coldstart-recsys/.notebook/a...


## Validation Summary

`FAIL` means later notebooks are not ready to run. `WARN` identifies expected
setup work that does not invalidate this notebook's inspection process.

In [15]:
SMOKE_STATUS = {record["check"]: record["status"] for record in SMOKE_CHECKS}
PATH_STATUS = {record["check"]: record["status"] for record in PATH_CHECKS}

VALIDATION_RESULTS = [
    {
        "requirement": "runtime context reported",
        "status": "PASS",
        "detail": f"{EXECUTION_CONTEXT} on Python {platform.python_version()}",
    },
    {
        "requirement": "valid project root",
        "status": PATH_STATUS.get("project_root_exists", "FAIL"),
        "detail": str(PROJECT_ROOT),
    },
    {
        "requirement": "valid reproducibility seed",
        "status": "FAIL" if SEED_CONFIGURATION_ERROR else "PASS",
        "detail": SEED_CONFIGURATION_ERROR or f"seed={DEFAULT_SEED}",
    },
    {
        "requirement": "required dependencies available",
        "status": "PASS" if not MISSING_REQUIRED else "FAIL",
        "detail": "none missing" if not MISSING_REQUIRED else ", ".join(MISSING_REQUIRED),
    },
    {
        "requirement": "root dependency manifest",
        "status": "PASS" if ROOT_DEPENDENCY_MANIFESTS else "WARN",
        "detail": (
            ", ".join(path.name for path in ROOT_DEPENDENCY_MANIFESTS)
            if ROOT_DEPENDENCY_MANIFESTS
            else "version locking remains a TODO"
        ),
    },
    {
        "requirement": "valid compute device",
        "status": "PASS" if DEVICE is not None else "FAIL",
        "detail": str(DEVICE) if DEVICE is not None else str(DEVICE_SELECTION_ERROR),
    },
    {
        "requirement": "Kaggle T4 accelerator contract",
        "status": KAGGLE_T4_STATUS,
        "detail": KAGGLE_T4_DETAIL,
    },
    {
        "requirement": "Torch dense smoke check",
        "status": SMOKE_STATUS.get("torch_dense", "FAIL"),
        "detail": "in-memory operation",
    },
    {
        "requirement": "Torch sparse smoke check",
        "status": SMOKE_STATUS.get("torch_sparse", "FAIL"),
        "detail": "in-memory operation",
    },
    {
        "requirement": "PyG message-passing smoke check",
        "status": SMOKE_STATUS.get("pyg_message_passing", "FAIL"),
        "detail": "in-memory GCNConv operation",
    },
    {
        "requirement": "repeatable seeded generators",
        "status": "PASS" if REPEATABILITY_CONFIRMED else "FAIL",
        "detail": REPEATABILITY_ERROR or f"seed={DEFAULT_SEED}",
    },
    {
        "requirement": "hash seed effective at kernel startup",
        "status": "PASS" if SEED_SETTINGS["python_hash_seed_matched_at_entry"] else "WARN",
        "detail": (
            f"PYTHONHASHSEED={DEFAULT_SEED}"
            if SEED_SETTINGS["python_hash_seed_matched_at_entry"]
            else (
                f"started with {SEED_SETTINGS['python_hash_seed_at_entry']!r}; "
                f"restart with PYTHONHASHSEED={DEFAULT_SEED}"
            )
        ),
    },
    {
        "requirement": "deterministic CUDA workspace",
        "status": (
            "PASS"
            if os.environ.get("CUBLAS_WORKSPACE_CONFIG") in VALID_CUBLAS_CONFIGURATIONS
            else "FAIL"
        ),
        "detail": os.environ.get("CUBLAS_WORKSPACE_CONFIG"),
    },
    {
        "requirement": "writable isolated artifact root",
        "status": (
            "PASS"
            if PATH_STATUS.get("artifact_root_writable") == "PASS"
            and PATH_STATUS.get("artifacts_outside_read_only_input") == "PASS"
            else "FAIL"
        ),
        "detail": str(ARTIFACT_ROOT),
    },
    {
        "requirement": "input root available",
        "status": PATH_STATUS.get("input_root_available", "WARN"),
        "detail": str(INPUT_ROOT),
    },
    {
        "requirement": "provenance and configuration hash captured",
        "status": "PASS" if CONFIGURATION_HASH else "FAIL",
        "detail": CONFIGURATION_HASH,
    },
    {
        "requirement": "notebook source hash captured",
        "status": "PASS" if NOTEBOOK_SOURCE_HASH else "WARN",
        "detail": NOTEBOOK_SOURCE_HASH or "source file unavailable in this runtime",
    },
]

ENVIRONMENT_READY = not any(
    result["status"] == "FAIL" for result in VALIDATION_RESULTS
)

show_records(VALIDATION_RESULTS, ["requirement", "status", "detail"])
display(
    Markdown(
        "### Environment readiness: " + ("PASS" if ENVIRONMENT_READY else "FAIL")
    )
)

,requirement,status,detail
0,runtime context reported,PASS,local on Python 3.12.3
1,valid project root,PASS,/workspace/HungPH/coldstart-recsys
2,valid reproducibility seed,PASS,seed=2025
3,required dependencies available,FAIL,"scipy, scikit-learn, torch-geometric, matplotl..."
4,root dependency manifest,WARN,version locking remains a TODO
5,valid compute device,PASS,cuda:0
6,Kaggle T4 accelerator contract,WARN,verified only when running on Kaggle
7,Torch dense smoke check,PASS,in-memory operation
8,Torch sparse smoke check,PASS,in-memory operation
9,PyG message-passing smoke check,PASS,in-memory GCNConv operation


### Environment readiness: PASS

## Deferred TODOs

- Select and commit a root dependency manifest after confirming a compatible
  Python, Torch, PyG, and CUDA version set.
- Move configuration and reproducibility helpers into `src/` with unit tests.
- Attach or provide MovieLens-1M before running notebook 01; do not substitute
  the repository's existing MovieLens-100K data.
- Persist provenance and artifacts only through the future shared run manager.
- Keep internet disabled unless a later notebook documents an explicit need.